In [ ]:
"""
test script to see if fred is accessible and can get the 3 macroeconomic indicators mentioned 
in transformer_options_pricing paper. 
"""

import dotenv
from fredapi import Fred
from datetime import datetime
import pandas as pd
import yfinance as yf

# Load the variables from the .env file into the environment
dotenv.load_dotenv()

# fredapi automatically grabs os.environ['FRED_API_KEY']
fred = Fred()

start_date = "2025-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

print("Pulling financial data...")

# 2. Fetch VIX and 10-Year Treasury from yfinance
# ^VIX = CBOE Volatility Index
# ^TNX = 10-Year Treasury Yield Index (multiplied by 10 internally, e.g., 42.5 = 4.25%)
yf_tickers = ["^VIX", "^TNX"]
market_data = yf.download(yf_tickers, start=start_date, end=end_date)["Close"]

# Format yfinance outputs
market_data = market_data.rename(
    columns={"^VIX": "VIX", "^TNX": "10Y_Treasury_Yield"}
)
# Divide ^TNX by 10 to yield the correct percentage decimal format (e.g., 4.25)
market_data["10Y_Treasury_Yield"] = market_data["10Y_Treasury_Yield"] / 10


# 3. Fetch Daily Inflation Expectations from FRED API
# T10YIE represents the daily 10-Year Breakeven Inflation Rate in percent
daily_inflation = fred.get_series("T10YIE", start_date, end_date)
inflation_df = pd.DataFrame(daily_inflation, columns=["Daily_Breakeven_Inflation"])


# 4. Alternative: Fetch Official Monthly Inflation (CPI)
# CPIAUCSL is the standard Consumer Price Index. It only updates once a month.
cpi_series = fred.get_series("CPIAUCSL", "2024-01-01", end_date)
cpi_df = pd.DataFrame(cpi_series, columns=["CPI"])
# Calculate the Year-over-Year (YoY) percentage shift
cpi_df["Official_Inflation_YoY"] = cpi_df["CPI"].pct_change(periods=12) * 100


# 5. Merge Datasets onto a Unified Calendar
daily_calendar = pd.date_range(start=start_date, end=end_date, freq="D")
master_df = pd.DataFrame(index=daily_calendar)

# Join the market indicators
master_df = master_df.join(market_data)
master_df = master_df.join(inflation_df)
master_df = master_df.join(cpi_df[["Official_Inflation_YoY"]])

# Clean up trading data gaps (forward-fill weekends and market holidays)
master_df["VIX"] = master_df["VIX"].ffill()
master_df["10Y_Treasury_Yield"] = master_df["10Y_Treasury_Yield"].ffill()
master_df["Daily_Breakeven_Inflation"] = (
    master_df["Daily_Breakeven_Inflation"].ffill()
)
master_df["Official_Inflation_YoY"] = master_df["Official_Inflation_YoY"].ffill()

# Drop rows before our 2025 boundary where data isn't fully calculated yet
master_df = master_df.loc["2025-01-01":]

print("\n--- Final Consolidated Dataset Sample ---")
print(master_df.tail(10))


[*********************100%***********************]  2 of 2 completed

Pulling financial data...



--- Final Consolidated Dataset Sample ---
            10Y_Treasury_Yield        VIX  Daily_Breakeven_Inflation  \
2026-09-08              0.4806  15.720000                       2.37   
2026-09-09              0.4837  16.459999                       2.37   
2026-09-10              0.4944  17.840000                       2.40   
2026-09-11              0.4975  15.840000                       2.36   
2026-09-12              0.4975  15.840000                       2.36   
2026-09-13              0.4975  15.840000                       2.36   
2026-09-14              0.4961  17.100000                       2.37   
2026-09-15              0.4996  17.200001                       2.38   
2026-09-16              0.5006  17.709999                       2.33   
2026-09-17              0.5006  17.709999                       2.33   

            Official_Inflation_YoY  
2026-09-08                3.353016  
2026-09-09                3.353016  
2026-09-10                3.353016  
2026-09-11      

C:\Users\joshf\AppData\Local\Temp\ipykernel_29140\2712046537.py:43: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  cpi_df["Official_Inflation_YoY"] = cpi_df["CPI"].pct_change(periods=12) * 100
